# 03 - DPO训练 (Direct Preference Optimization)

## 学习目标

本notebook深入讲解DPO直接偏好优化算法：

1. **DPO核心思想** - 为什么DPO比RLHF更简单
2. **数学推导** - 从RL目标到DPO损失的推导
3. **损失变体** - Sigmoid/Hinge/IPO对比
4. **训练实现** - 完整的DPO训练流程
5. **与RLHF对比** - 各维度详细对比

---

## 1. DPO理论基础

### 1.1 为什么需要DPO？

RLHF存在的问题：
- **复杂**: 需要训练奖励模型 + RL循环
- **不稳定**: PPO训练容易发散
- **超参数多**: 需要调整多个超参数
- **计算成本高**: 需要多次生成和评分

DPO的解决方案：
- **无需奖励模型**: 直接从偏好数据学习
- **无需RL训练**: 转化为监督学习问题
- **训练稳定**: 标准的交叉熵损失
- **超参数少**: 主要只有β超参数

### 1.2 核心洞察

DPO利用了最优策略的奖励函数满足以下性质：

$$r^*(x, y) = r_0(x, y) + \beta \log \frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + Z(x)$$

这意味着奖励函数与策略之间存在直接映射关系！

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Any, Optional

from dpo import (
    DPOConfig,
    DPOBatch,
    DPOTrainer,
    DPOLoss,
    compute_dpo_loss,
    compute_implicit_rewards
)

print("环境导入完成！")

## 2. DPO数学推导

### 2.1 从RL目标到DPO

**RL目标**:
$$\max_{\pi} \mathbb{E}_{x \sim D, y \sim \pi}[r(x, y)] - \beta \cdot \text{KL}(\pi || \pi_{ref})$$

**最优策略满足**:
$$\pi^*(y|x) \propto \pi_{ref}(y|x) \exp(\frac{1}{\beta} r(x, y))$$

**代入奖励函数**:
$$\frac{\pi^*(y|x)}{\pi_{ref}(y|x)} = \exp(\frac{1}{\beta}r(x, y)) \cdot \frac{1}{Z(x)}$$

取对数得到:
$$\log \frac{\pi^*(y|x)}{\pi_{ref}(y|x)} = \frac{1}{\beta}r(x, y) - \log Z(x)$$

因此:
$$r(x, y) = \beta \log \frac{\pi(y|x)}{\pi_{ref}(y|x)} + \beta \log Z(x)$$

### 2.2 DPO损失

将奖励函数代入Bradley-Terry模型：

$$L_{DPO} = -\mathbb{E}\left[\log \sigma\left(\beta \left(\log \frac{\pi(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right)\right]$$

In [ ]:
def compute_log_ratio(log_policy: np.ndarray, 
                       log_ref: np.ndarray) -> np.ndarray:
    """计算log ratio = log(π/π_ref)。"""
    return log_policy - log_ref

def dpo_loss_sigmoid(log_ratio_chosen: np.ndarray,
                     log_ratio_rejected: np.ndarray,
                     beta: float = 0.1) -> np.ndarray:
    """DPO损失 (Sigmoid版本)。"""
    # 计算logits差值
    logits = beta * (log_ratio_chosen - log_ratio_rejected)
    
    # Binary cross-entropy损失
    # -log(sigmoid(x)) = log(1 + exp(-x)) = softplus(-x)
    losses = np.log1p(np.exp(-logits))
    
    return losses

# 测试DPO损失计算
log_pi_chosen = np.array([-2.0, -1.5, -1.0, -0.5])
log_pi_ref_chosen = np.array([-3.0, -2.5, -2.0, -1.5])
log_pi_rejected = np.array([-2.5, -2.0, -1.5, -1.0])
log_pi_ref_rejected = np.array([-3.0, -2.5, -2.0, -1.5])

log_ratio_chosen = compute_log_ratio(log_pi_chosen, log_pi_ref_chosen)
log_ratio_rejected = compute_log_ratio(log_pi_rejected, log_pi_ref_rejected)

losses = dpo_loss_sigmoid(log_ratio_chosen, log_ratio_rejected, beta=0.1)

print("DPO损失计算:")
print("="*60)
for i, (rc, rl, loss) in enumerate(zip(log_ratio_chosen, log_ratio_rejected, losses)):
    print(f"样本 {i}: rc={rc:.3f}, rl={rl:.3f}, Δ={rc-rl:.3f}, loss={loss:.4f}")

In [ ]:
# 可视化DPO损失曲面
fig = plt.figure(figsize=(12, 5))

# 左图：不同beta下的损失
ax1 = fig.add_subplot(1, 2, 1)
ratios_chosen = np.linspace(-3, 3, 100)
ratios_rejected = np.zeros_like(ratios_chosen)

betas = [0.01, 0.1, 0.5, 1.0]
for beta in betas:
    losses = dpo_loss_sigmoid(ratios_chosen, ratios_rejected, beta)
    ax1.plot(ratios_chosen, losses, label=f'β={beta}')

ax1.set_xlabel('Log ratio差值', fontsize=12)
ax1.set_ylabel('DPO损失', fontsize=12)
ax1.set_title('不同β值下的损失函数', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 右图：2D损失曲面
ax2 = fig.add_subplot(1, 2, 2, projection='3d')

rc = np.linspace(-3, 3, 50)
rl = np.linspace(-3, 3, 50)
RC, RL = np.meshgrid(rc, rl)

beta = 0.1
diff = beta * (RC - RL)
Loss = np.log1p(np.exp(-diff))

surf = ax2.plot_surface(RC, RL, Loss, cmap='viridis', alpha=0.8)
ax2.set_xlabel('Log ratio (chosen)', fontsize=10)
ax2.set_ylabel('Log ratio (rejected)', fontsize=10)
ax2.set_zlabel('DPO Loss', fontsize=10)
ax2.set_title(f'DPO损失曲面 (β={beta})', fontsize=12)

plt.tight_layout()
plt.show()

print(f"\n关键观察:")
print(f"- β越大，损失对ratio差越敏感")
print(f"- 损失关于差值单调递减")
print(f"- 优化方向：增大chosen ratio，减小rejected ratio")

## 3. DPO损失变体对比

In [ ]:
def dpo_loss_hinge(log_ratio_chosen: np.ndarray,
                   log_ratio_rejected: np.ndarray,
                   beta: float = 0.1) -> np.ndarray:
    """DPO损失 (Hinge版本)。"""
    margin = beta * (log_ratio_chosen - log_ratio_rejected)
    return np.maximum(0, 1 - margin)

def dpo_loss_ipo(log_ratio_chosen: np.ndarray,
                 log_ratio_rejected: np.ndarray,
                 beta: float = 0.1) -> np.ndarray:
    """IPO (Identity Policy Optimization)损失。"""
    margin = beta * (log_ratio_chosen - log_ratio_rejected)
    return (margin - 1) ** 2

# 对比不同损失函数
ratios_chosen = np.array([-10.0, -8.0, -5.0, -2.0, 0.0, 2.0, 5.0])
ratios_rejected = np.array([-12.0, -10.0, -7.0, -4.0, -2.0, 0.0, 3.0])

diffs = ratios_chosen - ratios_rejected

losses_sigmoid = dpo_loss_sigmoid(ratios_chosen, ratios_rejected)
losses_hinge = dpo_loss_hinge(ratios_chosen, ratios_rejected)
losses_ipo = dpo_loss_ipo(ratios_chosen, ratios_rejected)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(diffs, losses_sigmoid, 'bo-', label='Sigmoid')
axes[0].set_xlabel('Log ratio差值')
axes[0].set_ylabel('损失')
axes[0].set_title('Sigmoid DPO')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(diffs, losses_hinge, 'ro-', label='Hinge')
axes[1].set_xlabel('Log ratio差值')
axes[1].set_ylabel('损失')
axes[1].set_title('Hinge DPO')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(diffs, losses_ipo, 'go-', label='IPO')
axes[2].set_xlabel('Log ratio差值')
axes[2].set_ylabel('损失')
axes[2].set_title('IPO')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print("\n损失对比:")
print("| 损失类型 | 特点 | 适用场景 |")
print("|---------|------|----------|")
print("| Sigmoid | 平滑，标准DPO | 大多数场景 |")
print("| Hinge | 鲁棒，边界清晰 | 噪声数据 |")
print("| IPO | 防止过拟合 | 小数据集 |")

## 4. DPO训练器配置与使用

In [ ]:
# 创建DPO训练器
config = DPOConfig(
    # DPO特定参数
    beta=0.1,              # DPO温度参数
    loss_type="sigmoid",   # 损失类型
    
    # 训练参数
    learning_rate=1e-6,
    batch_size=4,
    
    # 参考模型
    ref_model_freeze=True,  # 是否冻结参考模型
)

trainer = DPOTrainer(config)

print(f"DPO配置:")
print(f"  Beta: {config.beta}")
print(f"  损失类型: {config.loss_type}")
print(f"  学习率: {config.learning_rate}")
print(f"  批次大小: {config.batch_size}")

## 5. 准备偏好数据

In [ ]:
# 创建DPO批次数据
batch = DPOBatch(
    prompts=[
        "什么是深度学习？",
        "如何学习编程？",
        "Python的优点是什么？",
        "解释什么是过拟合",
    ],
    chosen_responses=[
        "深度学习是机器学习的一个分支，使用多层神经网络来学习数据的层次化表示。",
        "建议从Python开始，先学习基础语法，然后通过项目实践来提高。",
        "Python语法简洁、易学易用、生态丰富，适合新手和快速开发。",
        "过拟合是指模型在训练数据上表现很好，但在新数据上泛化能力差。",
    ],
    rejected_responses=[
        "不知道。",
        "去网上搜。",
        "不知道。",
        "不清楚。",
    ],
)

print(f"批次大小: {len(batch.prompts)}")
print(f"提示示例: {batch.prompts[0]}")
print(f"优选示例: {batch.chosen_responses[0][:40]}...")
print(f"拒绝示例: {batch.rejected_responses[0]}")

## 6. DPO训练循环

In [ ]:
# DPO训练
print("开始DPO训练...")
print("="*60)

history = {
    'loss': [],
    'accuracy': [],
    'reward_margin': [],
    'chosen_logp': [],
    'rejected_logp': [],
}

num_steps = 20

for step in range(num_steps):
    # 训练步骤
    metrics = trainer.train_step(batch)
    
    # 记录历史
    for key in history:
        if key in metrics:
            history[key].append(metrics[key])
    
    # 打印进度
    if (step + 1) % 5 == 0:
        print(f"Step {metrics['step']:2d}: "
              f"loss={metrics['loss']:6.3f}, "
              f"acc={metrics['accuracy']:5.2%}, "
              f"margin={metrics['reward_margin']:6.3f}")

print("\n训练完成！")
print(f"最终准确率: {history['accuracy'][-1]:.2%}")
print(f"最终奖励边距: {history['reward_margin'][-1]:.3f}")

In [ ]:
# 可视化训练过程
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 损失
axes[0, 0].plot(history['loss'], 'b-o')
axes[0, 0].set_title('DPO损失')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)

# 准确率
axes[0, 1].plot(history['accuracy'], 'g-o')
axes[0, 1].set_title('准确率')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].grid(True, alpha=0.3)

# 奖励边距
axes[1, 0].plot(history['reward_margin'], 'r-o')
axes[1, 0].set_title('奖励边距')
axes[1, 0].set_ylabel('Margin')
axes[1, 0].set_xlabel('Step')
axes[1, 0].grid(True, alpha=0.3)

# Log概率
axes[1, 1].plot(history['chosen_logp'], label='Chosen')
axes[1, 1].plot(history['rejected_logp'], label='Rejected')
axes[1, 1].set_title('Log概率')
axes[1, 1].set_ylabel('Log prob')
axes[1, 1].set_xlabel('Step')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 隐式奖励计算

In [ ]:
# 计算隐式奖励
implicit_rewards = trainer.compute_implicit_rewards(
    batch.prompts,
    batch.chosen_responses,
)

print(f"隐式奖励:")
print("="*40)
for i, (prompt, reward) in enumerate(zip(batch.prompts, implicit_rewards)):
    print(f"{i+1}. [{reward:.3f}] {prompt[:35]}...")

## 8. DPO vs RLHF 对比

In [ ]:
# 详细对比表格
comparison = [
    ("训练阶段", "3阶段", "1阶段"),
    ("奖励模型", "需要", "不需要"),
    ("RL训练", "需要(PPO)", "不需要"),
    ("稳定性", "较低", "较高"),
    ("超参数", "多", "少"),
    ("计算成本", "高", "低"),
    ("实现复杂度", "高", "低"),
    ("训练时间", "长", "短"),
]

print("\n="*70)
print(f"{'方面':<15} {'RLHF/PPO':<20} {'DPO':<20}")
print("="*70)
for aspect, rlhf, dpo in comparison:
    print(f"{aspect:<15} {rlhf:<20} {dpo:<20}")

In [ ]:
# 可视化对比
categories = ['训练阶段', '奖励模型', 'RL训练', '稳定性', 
             '超参数', '计算成本', '实现复杂度']
rlhf_scores = [3, 2, 1, 2, 1, 1, 1]  # 数值越高表示越复杂/成本越高
dpo_scores = [1, 2, 2, 3, 3, 3, 2]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width/2, rlhf_scores, width, label='RLHF/PPO', alpha=0.8)
bars2 = ax.bar(x + width/2, dpo_scores, width, label='DPO', alpha=0.8)

ax.set_xlabel('维度', fontsize=12)
ax.set_ylabel('评分 (越高越复杂/成本越高)', fontsize=12)
ax.set_title('DPO vs RLHF 对比', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.grid(True, alpha=0.3)

# 添加数值标签
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n关键优势:")
print(f"- DPO只需1阶段，简化训练流程")
print(f"- DPO无需训练单独的奖励模型")
print(f"- DPO训练更稳定，超参数更少")
print(f"- DPO计算成本更低")

## 9. Beta参数影响分析

In [ ]:
# 分析不同beta值的影响
betas = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
final_accuracies = []

for beta in betas:
    # 重新初始化训练器
    config = DPOConfig(beta=beta, learning_rate=1e-6)
    trainer = DPOTrainer(config)
    
    # 训练
    for _ in range(10):
        metrics = trainer.train_step(batch)
    
    final_accuracies.append(metrics['accuracy'])

# 可视化
plt.figure(figsize=(10, 5))
plt.plot(betas, final_accuracies, 'o-', linewidth=2, markersize=8)
plt.xlabel('Beta参数', fontsize=12)
plt.ylabel('最终准确率', fontsize=12)
plt.title('Beta参数对DPO性能的影响', fontsize=14)
plt.xscale('log')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nBeta参数建议:")
print(f"- β=0.01~0.05: 较弱的正则化，可能过拟合")
print(f"- β=0.1: 推荐值，平衡性能和稳定性")
print(f"- β=0.2~0.5: 强正则化，更保守")
print(f"- β>1.0: 过强的正则化，可能欠拟合")

## 总结

本notebook介绍了DPO直接偏好优化的核心内容：

### 关键要点

1. **核心思想**: 无需奖励模型，直接优化策略
2. **数学推导**: r ∝ log(π/π_ref)
3. **DPO损失**: -log σ(β(log(π_w/π_ref_w) - log(π_l/π_ref_l)))
4. **vs RLHF**: 更简单、更稳定、更快

### DPO超参数建议

| 参数 | 推荐值 | 说明 |
|------|--------|------|
| beta | 0.1 ~ 0.2 | 控制正则化强度 |
| learning_rate | 1e-6 ~ 5e-6 | 较小的学习率 |
| batch_size | 4 ~ 16 | 根据GPU内存调整 |

### 实践建议

1. **优先选择DPO**: 除非有特殊需求，DPO比RLHF更简单高效
2. **Beta调优**: 从0.1开始，根据验证集调整
3. **数据质量**: 偏好数据质量比数量更重要
4. **参考模型**: 保持参考模型固定，不更新

### 下一步

- 探索Constitutional AI（AI对齐）
- 了解更高级的对齐技术